# Multi-Task-LSTM ohne Kontext

## 1. Experiment-Konfiguration laden

Dieses Notebook bewertet den Multi-Task-LSTM ohne AEZ- oder Monatskontext.

## 2. Reproduzierbare Umgebung einrichten

Python-, Paket- und Git-Informationen werden vor dem Lauf angezeigt. Die Schätzer verwenden den Projekt-Seed.

## 3. Parameter und Eingabedaten definieren

`tune()` und `run()` laden ausschließlich den gemeinsamen Split und die festen CV-Folds. Das Notebook erzeugt weder eigene Splits noch eigene Merkmalsdateien.

## 4. Experiment ausführen

Zuerst wird ausschließlich die CV-Suche ausgeführt. Die Holdout-Zelle bleibt bis zur Entscheidung anhand der CV geschlossen.

## 5. Messwerte und Artefakte speichern

MLflow speichert Konfiguration, Kandidaten, Metriken, Confusion Matrix und Modell als Artefakte des Tuning- bzw. Bewertungslaufs.

## 6. Ergebnisse zur späteren Dokumentation exportieren

Die letzte Zelle zeigt eine kompakte Zusammenfassung mit IDs, Parametern, Metriken und Artefaktpfad für die spätere Projektdokumentation.

In [ ]:
import importlib.metadata
import platform
import subprocess

import numpy as np
import torch
from IPython.display import display

from awp2.config import LSTM_NO_CONTEXT, SEED
from awp2.experiment import TuneConfig, tune
from awp2.models import MultiTaskLSTMClassifier
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "multi_task_lstm_no_context"
TUNING_ID = f"{EXPERIMENT_ID}_search"
PREPROCESSING = PreprocessingConfig(
    use_meta=False,
    use_spectral_standard_scale=True,
)
PARAM_GRID = {
    "hidden_size": [32, 64],
    "num_layers": [1],
    "dropout": [0.2],
    "max_epochs": [50],
}

np.random.seed(SEED)
torch.manual_seed(SEED)
environment = {
    "python": platform.python_version(),
    "torch": importlib.metadata.version("torch"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "branch": subprocess.check_output(
        ["git", "branch", "--show-current"],
        text=True,
    ).strip(),
    "preprocessing": PREPROCESSING.model_dump(),
    "param_grid": PARAM_GRID,
}
display(environment)

tuned = tune(
    MultiTaskLSTMClassifier(context_mode=LSTM_NO_CONTEXT),
    TuneConfig(
        name=TUNING_ID,
        approach="multi-task",
        description=(
            "Multi-Task-LSTM ohne AEZ- oder Monatskontext; "
            "kleine Suche über die Encoderbreite."
        ),
        preprocessing=PREPROCESSING,
        param_grid=PARAM_GRID,
    ),
)
display(tuned.candidates.sort_values("rank"))
{"best_cv_score": tuned.best_score, "best_params": tuned.best_params, "tuning_run": tuned.run_id}

In [ ]:
from awp2.experiment import RunConfig, run

# Erst nach der CV-Entscheidung ausführen: genau eine Holdout-Bewertung.
result = run(
    tuned.best_model,
    RunConfig(
        name=EXPERIMENT_ID,
        approach="multi-task",
        description="CV-Gewinner des Multi-Task-LSTM ohne Kontext.",
        preprocessing=PREPROCESSING,
        tuning_run=tuned.run_id,
    ),
)
summary = {
    "experiment": EXPERIMENT_ID,
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "metrics": result.metrics.model_dump(),
    "model_uri": result.model_uri,
}
display(summary)